# 06 — Vanilla vs Gluten/Velox Benchmark

Run this notebook **twice**:
1. `make up` → run → saves `results_vanilla.json`
2. `make down && make up-gluten` → run → saves `results_gluten.json`

Cell 'Compare results' renders the chart once both files exist.


In [ ]:
import os, time
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *

GLUTEN_ENABLED = os.environ.get('GLUTEN_ENABLED', 'false').lower() == 'true'
MASTER   = 'spark://spark-master:7077'
DATA_DIR = '/workspace/data'

spark = (
    SparkSession.builder
    .appName('spark-notebook')
    .master(MASTER)
    .config('spark.executor.memory', '2g')
    .config('spark.driver.memory',   '1g')
    .config('spark.executor.cores',  '2')
    # Disable BypassMergeSortShuffleWriter — incompatible with Gluten columnar serializer
    .config('spark.shuffle.sort.bypassMergeThreshold', '0')
    .getOrCreate()
)
spark.sparkContext.setLogLevel('WARN')
mode = 'Gluten/Velox' if GLUTEN_ENABLED else 'Vanilla'
print(f'Spark {spark.version}  |  Mode: {mode}')
spark.range(3).show()


## Load benchmark data

In [ ]:
from pathlib import Path
import json, datetime

RESULTS_DIR = '/workspace/notebooks/results'
Path(RESULTS_DIR).mkdir(exist_ok=True)
MODE = 'gluten' if GLUTEN_ENABLED else 'vanilla'

lineitem = spark.read.parquet(f'{DATA_DIR}/lineitem.parquet')
orders   = spark.read.parquet(f'{DATA_DIR}/orders.parquet')
customer = spark.read.parquet(f'{DATA_DIR}/customer.parquet')

lineitem.createOrReplaceTempView('lineitem')
orders.createOrReplaceTempView('orders')
customer.createOrReplaceTempView('customer')
print(f'Mode: {MODE} | lineitem: {lineitem.count():,} | orders: {orders.count():,}')


## Benchmark queries (TPC-H subset)

In [ ]:
QUERIES = {
    "Q1_agg": """
        SELECT l_returnflag, l_linestatus,
               SUM(l_quantity) AS sum_qty,
               SUM(l_extendedprice) AS sum_base_price,
               SUM(l_extendedprice * (1-l_discount)) AS sum_disc_price,
               SUM(l_extendedprice * (1-l_discount) * (1+l_tax)) AS sum_charge,
               AVG(l_quantity) AS avg_qty,
               COUNT(*) AS count_order
        FROM lineitem
        WHERE l_shipdate <= date '1998-09-01'
        GROUP BY l_returnflag, l_linestatus
        ORDER BY l_returnflag, l_linestatus
    """,
    "Q3_join": """
        SELECT l_orderkey, SUM(l_extendedprice*(1-l_discount)) AS revenue,
               o_orderdate, o_orderpriority
        FROM customer JOIN orders ON c_custkey = o_custkey
                      JOIN lineitem ON l_orderkey = o_orderkey
        WHERE o_orderdate < date '1995-03-15'
          AND l_shipdate  > date '1995-03-15'
        GROUP BY l_orderkey, o_orderdate, o_orderpriority
        ORDER BY revenue DESC LIMIT 10
    """,
    "Q6_filter": """
        SELECT SUM(l_extendedprice * l_discount) AS revenue
        FROM lineitem
        WHERE l_shipdate >= date '1994-01-01'
          AND l_shipdate  < date '1995-01-01'
          AND l_discount BETWEEN 0.05 AND 0.07
          AND l_quantity < 24
    """,
    "Q_window": """
        SELECT l_orderkey,
               SUM(l_extendedprice) OVER (PARTITION BY l_returnflag ORDER BY l_shipdate) AS running_total
        FROM lineitem LIMIT 1000
    """,
}
WARMUP = "Q6_filter"
print("Queries ready:", list(QUERIES.keys()))


# Q_window uses shuffle with columnar serializer incompatible in Gluten 1.6.0 + Spark 4.0
# Remove it from Gluten runs to avoid UnsupportedOperationException
if GLUTEN_ENABLED:
    QUERIES.pop('Q_window', None)
    print('NOTE: Q_window skipped in Gluten mode (shuffle serializer incompatibility in Gluten 1.6.0)')


## Warmup + Run

In [ ]:
def run_query(name, sql, runs=3):
    times = []
    for _ in range(runs):
        t0 = time.time()
        spark.sql(sql).collect()
        times.append(round(time.time()-t0, 3))
    return {"query": name, "mode": MODE, "times": times,
            "median": round(sorted(times)[len(times)//2], 3),
            "ts": datetime.datetime.now().isoformat()}


print('Warming up...')
spark.sql(QUERIES[WARMUP]).collect()

results = []
for name, sql in QUERIES.items():
    print(f'Running {name}...', end=' ')
    try:
        r = run_query(name, sql)
        print(f'median={r["median"]}s')
        results.append(r)
    except Exception as e:
        print(f'SKIPPED ({type(e).__name__}: {str(e)[:80]})')

out = f'{RESULTS_DIR}/results_{MODE}.json'
Path(out).write_text(json.dumps(results, indent=2))
print(f'Saved → {out}')


## Verify Gluten offloading

In [ ]:
if GLUTEN_ENABLED:
    plan = spark.sql(QUERIES["Q1_agg"]).explain(True)
    print("Look for 'Velox' prefixed operators in the plan above.")
else:
    print("Vanilla mode — no Gluten operators.")


## Compare results

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

v_file = Path(f"{RESULTS_DIR}/results_vanilla.json")
g_file = Path(f"{RESULTS_DIR}/results_gluten.json")

if not v_file.exists() or not g_file.exists():
    print("Run both vanilla and gluten passes first.")
else:
    v = {r["query"]: r["median"] for r in json.loads(v_file.read_text())}
    g = {r["query"]: r["median"] for r in json.loads(g_file.read_text())}

    # Only compare queries present in BOTH results
    # (Q_window is skipped in Gluten mode due to shuffle serializer incompatibility)
    queries = [q for q in v.keys() if q in g]
    skipped = [q for q in v.keys() if q not in g]
    if skipped:
        print(f"NOTE: {skipped} skipped in Gluten mode — excluded from comparison")

    speedups = [round(v[q]/g[q], 2) for q in queries]

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    x = np.arange(len(queries))
    w = 0.35
    axes[0].bar(x-w/2, [v[q] for q in queries], w, label='Vanilla', color='steelblue')
    axes[0].bar(x+w/2, [g[q] for q in queries], w, label='Gluten', color='darkorange')
    axes[0].set_xticks(x); axes[0].set_xticklabels(queries, rotation=20)
    axes[0].set_ylabel('seconds (median)'); axes[0].set_title('Query Time: Vanilla vs Gluten')
    axes[0].legend()

    colors = ['green' if s>=1 else 'red' for s in speedups]
    axes[1].bar(queries, speedups, color=colors)
    axes[1].axhline(1.0, color='black', linestyle='--')
    axes[1].set_ylabel('speedup (vanilla/gluten)'); axes[1].set_title('Gluten Speedup (>1 = faster)')
    for i,(q,s) in enumerate(zip(queries, speedups)):
        axes[1].text(i, s+0.02, f'{s}x', ha='center', fontsize=9)

    plt.tight_layout()
    plt.savefig(f"{RESULTS_DIR}/benchmark.png", dpi=150)
    plt.show()
    print("Speedups:", dict(zip(queries, speedups)))
